In [1]:
import pandas as pd

# New Google Drive file ID from your updated link
file_id = '1BHmQEffT6SMO1aIW2nEBB-6sqQGoX9_n'
download_url = f'https://docs.google.com/uc?export=download&id={file_id}'
file_path = '/content/SML_PrimaryPreprocessed_New.xlsx'

print("Downloading the newly cleaned dataset...")
!gdown {file_id} -O {file_path}

try:
    # Reading the new dataset using pandas
    df = pd.read_excel(file_path)
    print("New dataset successfully loaded!")
    display(df.head())
except Exception as e:
    print(f"Error reading the new Excel file: {e}")

Downloading...
From: https://drive.google.com/uc?id=1BHmQEffT6SMO1aIW2nEBB-6sqQGoX9_n
To: /content/SML_PrimaryPreprocessed_New.xlsx
100% 68.0k/68.0k [00:00<00:00, 52.0MB/s]
New dataset successfully loaded!


,ScenarioID,density_tier,meeting_length_min,meeting_modality,meeting_type,sched_mon_0700,sched_mon_0800,sched_mon_0900,sched_mon_1000,sched_mon_1100,...,avoid_time_2224,Preferred Modality,Back-To-Back Tolerance,Preferred Break,First Target Day,First Target Time,Second Target Day,Second Target Time,Third Target Day,Third Target Time
0,SC001,Light,60,Onsite,Meeting,False,False,False,False,False,...,0,Onsite,2,60,Tuesday,800,Thursday,1000,Friday,800
1,SC001,Light,60,Onsite,Meeting,False,False,False,False,False,...,0,Online,3,60,Friday,1100,Monday,1200,Tuesday,1200
2,SC001,Light,60,Onsite,Meeting,False,False,False,False,False,...,0,Onsite,2,90,Friday,1500,Friday,900,Wednesday,1600
3,SC002,Heavy,120,Online,Event,False,False,False,False,ON,...,0,Online,2,60,Wednesday,800,Tuesday,1200,Thursday,900
4,SC002,Heavy,120,Online,Event,False,False,False,False,ON,...,0,Onsite,2,90,Monday,1600,Thursday,900,Wednesday,1900


## Step 1: Cleaning & Feature Engineering
We will first clean column names, identify unique respondents, use a union-find structure to build Cross-Validation groups, and parse messy target times to valid start hours.

In [2]:
import numpy as np
import pandas as pd
import re
import datetime
import networkx as nx

# 1. Strip column-name whitespace
df.columns = [c.strip() for c in df.columns]

# 2. Respondent proxy
pref_cols = [
    'Commute', 'Preferred Modality', 'Back-To-Back Tolerance', 'Preferred Break',
    'PrefMon', 'PrefTue', 'PrefWed', 'PrefThurs', 'PrefFri', 'PrefSat',
    'AvoidMon', 'AvoidTue', 'AvoidWed', 'AvoidThurs', 'AvoidFri', 'AvoidSat',
    'pref_time_0708', 'pref_time_0810', 'pref_time_1012', 'pref_time_1214',
    'pref_time_1416', 'pref_time_1618', 'pref_time_1820', 'pref_time_2022', 'pref_time_2224',
    'avoid_time_0708', 'avoid_time_0810', 'avoid_time_1012', 'avoid_time_1214',
    'avoid_time_1416', 'avoid_time_1618', 'avoid_time_1820', 'avoid_time_2022', 'avoid_time_2224'
]

# Factorize the concatenated preference columns to get respondent_id
df['respondent_str'] = df[pref_cols].astype(str).agg('-'.join, axis=1)
df['respondent_id'] = pd.factorize(df['respondent_str'])[0]

# 3. CV groups using NetworkX for finding connected components
G = nx.Graph()
for i, row in df.iterrows():
    scenario_node = f"S_{row['ScenarioID']}"
    respondent_node = f"R_{row['respondent_id']}"
    G.add_edge(scenario_node, respondent_node)

# Map each node to its connected component ID
components = list(nx.connected_components(G))
node_to_group = {}
for group_idx, component in enumerate(components):
    for node in component:
        node_to_group[node] = group_idx

df['cv_group'] = df['ScenarioID'].apply(lambda x: node_to_group[f"S_{x}"])

# 4. Parse target times to start hour
def parse_target_time(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        if val < 1.0:
            # Excel fraction of day
            hours = val * 24
            # round to 15 min
            return round(hours * 4) / 4
        else:
            # e.g., 1000 -> 10.0
            val_str = str(int(val))
    else:
        val_str = str(val).strip()

    # Check range like "14:00 - 17:00"
    if '-' in val_str:
        val_str = val_str.split('-')[0].strip()
    # Check list like "1400, 1600"
    if ',' in val_str:
        val_str = val_str.split(',')[0].strip()
    if ';' in val_str:
        val_str = val_str.split(';')[0].strip()

    val_str = val_str.replace(':', '')
    # extract digits
    digits = re.sub(r'\D', '', val_str)
    if not digits:
        return np.nan

    if len(digits) <= 2:
        hr = int(digits)
        mn = 0
    else:
        hr = int(digits[:-2])
        mn = int(digits[-2:])

    return hr + mn / 60.0

# Parse commute
df['commute_min'] = df['Commute'].apply(lambda x: x * 60 if x <= 4 else x)

print(f"Responses: {len(df)}")
print(f"Unique Scenarios: {df['ScenarioID'].nunique()}")
print(f"Unique Respondent Profiles: {df['respondent_id'].nunique()}")
print(f"CV Groups: {df['cv_group'].nunique()}")

Responses: 110
Unique Scenarios: 51
Unique Respondent Profiles: 82
CV Groups: 26


## Step 2: Candidates and Graded Relevance Generation
We expand each query (response row) into feasible candidate slots (day, start_hour) based on schedule occupancy, and match them with the three target labels to construct training/evaluation instances.

In [3]:
days = ['mon', 'tue', 'wed', 'thu', 'fri', 'sat']
day_names_map = {
    'monday': 'mon', 'tuesday': 'tue', 'wednesday': 'wed', 'thursday': 'thu', 'friday': 'fri', 'saturday': 'sat',
    'mon': 'mon', 'tue': 'tue', 'wed': 'wed', 'thu': 'thu', 'fri': 'fri', 'sat': 'sat'
}

candidates_list = []
labels_kept = 0
labels_outside_grid = 0
labels_occupied = 0
valid_queries_count = 0

for idx, row in df.iterrows():
    L = int(row['meeting_length_min']) // 60

    # Parse labels
    targets = []
    for rank, target_day_col, target_time_col, rel in [('First', 'First Target Day', 'First Target Time', 3),
                                                       ('Second', 'Second Target Day', 'Second Target Time', 2),
                                                       ('Third', 'Third Target Day', 'Third Target Time', 1)]:
        t_day_val = row.get(target_day_col)
        t_time_val = row.get(target_time_col)
        if pd.isna(t_day_val) or pd.isna(t_time_val):
            continue

        day_clean = str(t_day_val).strip().lower()
        day_key = day_names_map.get(day_clean)
        hr = parse_target_time(t_time_val)

        if not day_key or pd.isna(hr):
            continue

        snapped_hr = int(np.floor(hr))

        # Check validation criteria
        if snapped_hr < 7 or snapped_hr + L > 18:
            labels_outside_grid += 1
            continue

        # Check occupancy
        occupied = False
        for h in range(snapped_hr, snapped_hr + L):
            col_name = f"sched_{day_key}_{h:02d}00"
            if col_name in row and row[col_name] in ['ON', 'OL']:
                occupied = True
                break

        if occupied:
            labels_occupied += 1
            continue

        targets.append({'day': day_key, 'start_hour': snapped_hr, 'relevance': rel})
        labels_kept += 1

    # Create unique labels (keep highest score if overlapping)
    target_map = {}
    for tg in targets:
        key = (tg['day'], tg['start_hour'])
        target_map[key] = max(target_map.get(key, 0), tg['relevance'])

    # Generate all candidate slots
    query_candidates = []
    for day in days:
        for s in range(7, 18 - L + 1):
            # check if all blocks are free
            occupied = False
            for h in range(s, s + L):
                col_name = f"sched_{day}_{h:02d}00"
                if col_name in row and row[col_name] in ['ON', 'OL']:
                    occupied = True
                    break
            if not occupied:
                rel = target_map.get((day, s), 0)
                query_candidates.append({
                    'query_idx': idx,
                    'ScenarioID': row['ScenarioID'],
                    'cv_group': row['cv_group'],
                    'day': day,
                    'start_hour': s,
                    'end_hour': s + L,
                    'relevance': rel
                })

    if len(query_candidates) > 0 and sum(target_map.values()) > 0:
        valid_queries_count += 1
        candidates_list.extend(query_candidates)

cand_df = pd.DataFrame(candidates_list)

print(f"Sanity Checks:")
print(f"- Candidates generated: {len(cand_df)}")
print(f"- Labels kept: {labels_kept}")
print(f"- Dropped as outside grid: {labels_outside_grid}")
print(f"- Dropped as occupied: {labels_occupied}")
print(f"- Queries with at least one usable label: {valid_queries_count}")

Sanity Checks:
- Candidates generated: 4982
- Labels kept: 286
- Dropped as outside grid: 21
- Dropped as occupied: 23
- Queries with at least one usable label: 101


## Step 3: Feature Engineering (33 features)
Now, we compute all 33 features per candidate slot using the main DataFrame's context.

In [ ]:
features_list = []

# Precompute schedule grids and day characteristics per row
for cand_idx, cand in cand_df.iterrows():
    row = df.loc[cand['query_idx']]
    day = cand['day']
    s = cand['start_hour']
    e = cand['end_hour']
    L = e - s

    # Slot features
    day_idx = days.index(day)
    is_saturday = 1.0 if day == 'sat' else 0.0
    overlaps_lunch = 1.0 if (s < 13 and e > 12) else 0.0
    is_morning = 1.0 if s < 12 else 0.0
    is_late = 1.0 if e > 16 else 0.0

    # Find classes context on that day
    class_hours = []
    class_types = []
    for h in range(7, 18):
        col = f"sched_{day}_{h:02d}00"
        val = row.get(col, False)
        if val in ['ON', 'OL']:
            class_hours.append(h)
            class_types.append(val)

    # Gap before and after
    gap_before = 6.0
    gap_after = 6.0
    classes_before = [ch for ch in class_hours if ch < s]
    classes_after = [ch for ch in class_hours if ch >= e]

    if classes_before:
        gap_before = min(6.0, s - max(classes_before) - 1)
    if classes_after:
        gap_after = min(6.0, min(classes_after) - e)

    min_gap = min(gap_before, gap_after)
    back_to_back = 1.0 if (gap_before == 0 or gap_after == 0) else 0.0

    # Day statistics
    day_class_hours = float(len(class_hours))
    day_is_free = 1.0 if day_class_hours == 0 else 0.0

    # Determine if it's the busiest day
    weekly_class_hours = [sum([1 for h in range(7, 18) if row.get(f"sched_{d}_{h:02d}00") in ['ON', 'OL']]) for d in days]
    max_weekly_hours = max(weekly_class_hours)
    day_is_busiest = 1.0 if (day_class_hours == max_weekly_hours and day_class_hours > 0) else 0.0

    day_onsite_hours = float(sum([1 for ct in class_types if ct == 'ON']))

    # Preferences match
    pref_day_col = 'Pref' + day.capitalize() if day != 'thu' else 'PrefThurs'
    avoid_day_col = 'Avoid' + day.capitalize() if day != 'thu' else 'AvoidThurs'
    pref_day = float(row.get(pref_day_col, 0))
    avoid_day = float(row.get(avoid_day_col, 0))

    # Preference/Avoid times
    bins = [
        (7, 8, '0708'), (8, 10, '0810'), (10, 12, '1012'), (12, 14, '1214'),
        (14, 16, '1416'), (16, 18, '1618'), (18, 20, '1820'), (20, 22, '2022'), (22, 24, '2224')
    ]

    pref_time_share = 0.0
    avoid_time_share = 0.0
    for b_start, b_end, b_lbl in bins:
        # Overlap of [s, e] and [b_start, b_end]
        overlap = max(0.0, min(e, b_end) - max(s, b_start))
        if overlap > 0:
            p_flag = float(row.get(f"pref_time_{b_lbl}", 0))
            a_flag = float(row.get(f"avoid_time_{b_lbl}", 0))
            pref_time_share += (overlap / L) * p_flag
            avoid_time_share += (overlap / L) * a_flag

    pref_break = float(row.get('Preferred Break', 0))
    break_shortfall = max(0.0, (pref_break / 60.0) - min_gap) if day_class_hours > 0 else 0.0

    b2b_tolerance = float(row.get('Back-To-Back Tolerance', 0))
    b2b_x_tolerance = back_to_back * (6.0 - b2b_tolerance)

    meeting_mod = row['meeting_modality']
    pref_mod = row['Preferred Modality']
    modality_match = 1.0 if pref_mod == meeting_mod else 0.0
    pref_modality_none = 1.0 if pref_mod == 'No Preference' else 0.0

    commute_min = float(row['commute_min'])
    extra_trip_cost = commute_min if (meeting_mod == 'Onsite' and day_onsite_hours == 0) else 0.0

    # Check if touching class is online
    online_next_to_class = 0.0
    if gap_before == 0 and len(classes_before) > 0:
        idx_before = class_hours.index(max(classes_before))
        if class_types[idx_before] == 'OL':
            online_next_to_class = 1.0
    if gap_after == 0 and len(classes_after) > 0:
        idx_after = class_hours.index(min(classes_after))
        if class_types[idx_after] == 'OL':
            online_next_to_class = 1.0

    features_list.append({
        'day_idx': day_idx,
        'is_saturday': is_saturday,
        'overlaps_lunch': overlaps_lunch,
        'is_morning': is_morning,
        'is_late': is_late,
        'gap_before': gap_before,
        'gap_after': gap_after,
        'min_gap': min_gap,
        'back_to_back': back_to_back,
        'day_class_hours': day_class_hours,
        'day_is_free': day_is_free,
        'day_is_busiest': day_is_busiest,
        'day_onsite_hours': day_onsite_hours,
        'meeting_len_h': float(L),
        'meeting_onsite': 1.0 if meeting_mod == 'Onsite' else 0.0,
        'type_project': 1.0 if row['meeting_type'] == 'Project' else 0.0,
        'type_event': 1.0 if row['meeting_type'] == 'Event' else 0.0,
        'tier_heavy': 1.0 if row['density_tier'] == 'Heavy' else 0.0,
        'tier_medium': 1.0 if row['density_tier'] == 'Medium' else 0.0,
        'n_feasible': float(row['n_feasible_slots']),
        'total_class_hours': float(row['total_class_hours']),
        'pref_day': pref_day,
        'avoid_day': avoid_day,
        'pref_time_share': pref_time_share,
        'avoid_time_share': avoid_time_share,
        'break_shortfall': break_shortfall,
        'b2b_tolerance': b2b_tolerance,
        'b2b_x_tolerance': b2b_x_tolerance,
        'modality_match': modality_match,
        'pref_modality_none': pref_modality_none,
        'commute_min': commute_min,
        'extra_trip_cost': extra_trip_cost,
        'online_next_to_class': online_next_to_class
    })

feat_df = pd.DataFrame(features_list)
# Align features index with cand_df
final_df = pd.concat([cand_df, feat_df], axis=1)
feature_names = list(feat_df.columns)
print(f"Extracted {len(feature_names)} features successfully!")

Extracted 33 features successfully!


## Step 4: Model Definitions & Evaluation
We train the ML models (Logistic Regression, Random Forest, Gradient Boosting Ranker, and optional LightGBM Ranker), as well as reference baselines, using 5-fold GroupKFold validation.

In [5]:
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingRegressor
import joblib

# Try to import LightGBM gracefully
try:
    import lightgbm as lgb
    LGBM_AVAILABLE = True
except ImportError:
    LGBM_AVAILABLE = False
    print("LightGBM is not installed. Skipping LightGBM gracefully.")

# Helper to compute ranking metrics
def ndcg_at_k(relevances, k):
    if sum(relevances) == 0:
        return 0.0
    # ideal sorted relevances
    idcg = sum([(2**rel - 1) / np.log2(idx + 2) for idx, rel in enumerate(sorted(relevances, reverse=True)[:k])])
    if idcg == 0:
        return 0.0
    dcg = sum([(2**rel - 1) / np.log2(idx + 2) for idx, rel in enumerate(relevances[:k])])
    return dcg / idcg

def mrr_score(relevances):
    for idx, rel in enumerate(relevances):
        if rel > 0:
            return 1.0 / (idx + 1)
    return 0.0

def evaluate_predictions(df_eval, score_col):
    # Sort candidate rows within each query
    df_sorted = df_eval.sort_values(by=['query_idx', score_col], ascending=[True, False])

    ndcg1, ndcg3, ndcg5, hit1, hit3, mrr = [], [], [], [], [], []

    for q_idx, group in df_sorted.groupby('query_idx'):
        rels = group['relevance'].values
        ndcg1.append(ndcg_at_k(rels, 1))
        ndcg3.append(ndcg_at_k(rels, 3))
        ndcg5.append(ndcg_at_k(rels, 5))
        hit1.append(1.0 if any(r > 0 for r in rels[:1]) else 0.0)
        hit3.append(1.0 if any(r > 0 for r in rels[:3]) else 0.0)
        mrr.append(mrr_score(rels))

    return {
        'NDCG@1': np.mean(ndcg1),
        'NDCG@3': np.mean(ndcg3),

        'NDCG@5': np.mean(ndcg5),
        'Hit@1': np.mean(hit1),
        'Hit@3': np.mean(hit3),
        'MRR': np.mean(mrr)
    }

# Rule-based score logic
def compute_rule_based(row):
    return (2 * row['pref_day'] - 2 * row['avoid_day'] + 2 * row['pref_time_share']
            - 2 * row['avoid_time_share'] - 1.5 * row['break_shortfall']
            - 0.5 * row['b2b_x_tolerance'] * row['back_to_back'] + 1.0 * row['modality_match']
            - row['extra_trip_cost']/60.0 - 0.5 * row['overlaps_lunch'] - 0.5 * row['is_saturday'])

In [6]:
# Cross Validation setup
gkf = GroupKFold(n_splits=5)
groups = final_df['cv_group'].values
queries = final_df['query_idx'].values

# Initialize lists to store out-of-fold predictions
final_df['pred_lr'] = 0.0
final_df['pred_rf'] = 0.0
final_df['pred_gb'] = 0.0
final_df['pred_lgb'] = 0.0
final_df['pred_pop'] = 0.0
final_df['pred_rand'] = np.random.default_rng(42).random(len(final_df))
final_df['pred_rule'] = final_df.apply(compute_rule_based, axis=1)

fold_metrics = []

for fold, (train_idx, test_idx) in enumerate(gkf.split(final_df, final_df['relevance'], groups)):
    X_train, y_train = final_df.loc[train_idx, feature_names], final_df.loc[train_idx, 'relevance']
    X_test, y_test = final_df.loc[test_idx, feature_names], final_df.loc[test_idx, 'relevance']

    # Targets for classifiers
    y_train_bin = (y_train > 0).astype(int)

    # 1. Logistic Regression
    lr = Pipeline([('scaler', StandardScaler()), ('lr', LogisticRegression(C=0.5, class_weight='balanced', max_iter=2000, random_state=42))])
    lr.fit(X_train, y_train_bin)
    final_df.loc[test_idx, 'pred_lr'] = lr.predict_proba(X_test)[:, 1]

    # 2. Random Forest
    rf = RandomForestClassifier(n_estimators=400, max_depth=8, min_samples_leaf=5, max_features='sqrt', class_weight='balanced_subsample', n_jobs=-1, random_state=42)
    rf.fit(X_train, y_train_bin)
    final_df.loc[test_idx, 'pred_rf'] = rf.predict_proba(X_test)[:, 1]

    # 3. Gradient Boosting Ranker
    gb = GradientBoostingRegressor(n_estimators=250, learning_rate=0.05, max_depth=3, subsample=0.8, min_samples_leaf=10, random_state=42)
    gb.fit(X_train, y_train)
    final_df.loc[test_idx, 'pred_gb'] = gb.predict(X_test)

    # 4. Popularity Baseline
    pop_map = final_df.loc[train_idx].groupby(['day', 'start_hour'])['relevance'].sum().to_dict()
    final_df.loc[test_idx, 'pred_pop'] = final_df.loc[test_idx].apply(lambda r: pop_map.get((r['day'], r['start_hour']), 0.0), axis=1)

    # 5. Optional LightGBM
    if LGBM_AVAILABLE:
        # Sort train data by queries for lambdarank
        train_df_fold = final_df.loc[train_idx].sort_values('query_idx')
        q_sizes_train = train_df_fold.groupby('query_idx').size().values
        X_tr_lgb = train_df_fold[feature_names]
        y_tr_lgb = train_df_fold['relevance']

        lgb_model = lgb.LGBMRanker(objective='lambdarank', n_estimators=200, learning_rate=0.05, num_leaves=7, min_child_samples=10, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=42, n_jobs=-1, verbose=-1)
        lgb_model.fit(X_tr_lgb, y_tr_lgb, group=q_sizes_train)
        final_df.loc[test_idx, 'pred_lgb'] = lgb_model.predict(X_test)

print("Cross-validation completed successfully!")

Cross-validation completed successfully!


## Step 5: Metric Evaluation Tables & Feature Importances
Let's print pooled out-of-fold performance metrics and display the top 8 feature importances.

In [7]:
models = ['pred_rand', 'pred_pop', 'pred_rule', 'pred_lr', 'pred_rf', 'pred_gb']
if LGBM_AVAILABLE:
    models.append('pred_lgb')

metrics_results = {}
for m in models:
    metrics_results[m] = evaluate_predictions(final_df, m)

metrics_df = pd.DataFrame(metrics_results).T
display(metrics_df)

# Save candidate rows and metrics
final_df.to_csv('candidate_predictions.csv', index=False)

# Refit on all data & save
rf_final = RandomForestClassifier(n_estimators=400, max_depth=8, min_samples_leaf=5, max_features='sqrt', class_weight='balanced_subsample', n_jobs=-1, random_state=42)
rf_final.fit(final_df[feature_names], (final_df['relevance'] > 0).astype(int))
joblib.dump(rf_final, 'random_forest_model.joblib')

gb_final = GradientBoostingRegressor(n_estimators=250, learning_rate=0.05, max_depth=3, subsample=0.8, min_samples_leaf=10, random_state=42)
gb_final.fit(final_df[feature_names], final_df['relevance'])
joblib.dump(gb_final, 'gradient_boosting_model.joblib')

# Feature Importances
importances_rf = pd.Series(rf_final.feature_importances_, index=feature_names).sort_values(ascending=False)
importances_gb = pd.Series(gb_final.feature_importances_, index=feature_names).sort_values(ascending=False)

print("\nTop 8 Features for Random Forest:")
print(importances_rf.head(8))
print("\nTop 8 Features for Gradient Boosting:")
print(importances_gb.head(8))

,NDCG@1,NDCG@3,NDCG@5,Hit@1,Hit@3,MRR
pred_rand,0.026874,0.055284,0.074966,0.049505,0.158416,0.177229
pred_pop,0.104668,0.158708,0.207220,0.118812,0.396040,0.315336
pred_rule,0.207921,0.231831,0.275303,0.247525,0.415842,0.388173
pred_lr,0.264498,0.300141,0.348838,0.386139,0.633663,0.548823
pred_rf,0.240453,0.299666,0.360161,0.396040,0.613861,0.548308
pred_gb,0.240453,0.296838,0.353651,0.376238,0.594059,0.525299
pred_lgb,0.226308,0.298539,0.347292,0.336634,0.613861,0.517627



Top 8 Features for Random Forest:
day_idx              0.106078
min_gap              0.101996
gap_before           0.076468
is_saturday          0.066747
pref_time_share      0.050915
n_feasible           0.050520
gap_after            0.047612
total_class_hours    0.047535
dtype: float64

Top 8 Features for Gradient Boosting:
gap_before         0.123507
min_gap            0.118015
day_idx            0.088601
break_shortfall    0.072263
pref_time_share    0.060184
is_late            0.057769
gap_after          0.052921
is_morning         0.047946
dtype: float64


## Step 6: Inference Helper & Downloads
We provide a clean inference function that takes a raw response row and predicts/ranks the top 5 feasible slots, followed by a download trigger for all generated files.

In [8]:
from google.colab import files

def get_top_slots(response_row, model_type='gb', top_n=5):
    # Extract candidates and features for this single row
    query_id = response_row.name
    sub_df = final_df[final_df['query_idx'] == query_id].copy()

    if sub_df.empty:
        return "No feasible candidates found."

    score_col = f'pred_{model_type}'
    sub_df = sub_df.sort_values(by=score_col, ascending=False)
    return sub_df[['day', 'start_hour', 'end_hour', score_col]].head(top_n)

# Test on the first query row
test_row = df.iloc[0]
print("Inference Demo for Response 0:")
display(get_top_slots(test_row))

# Download artifacts
print("Downloading artifacts...")
for file_name in ['candidate_predictions.csv', 'random_forest_model.joblib', 'gradient_boosting_model.joblib']:
    try:
        files.download(file_name)
    except Exception:
        pass

Inference Demo for Response 0:


,day,start_hour,end_hour,pred_gb
45,fri,15,16,0.474219
1,mon,8,9,0.414310
0,mon,7,8,0.414310
2,mon,9,10,0.414310
41,fri,9,10,0.320024


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>